# Dual-Domain Frequency GNN + Spatial CNN for JPEG Artifact Removal
### DIP Project: Predicting High-Frequency DCT Harmonics with Multi-Block Spatial Context (1+2+5 Approach)

> **Step 0: Enable GPU Acceleration**
> Before running, ensure GPU is enabled: Click **Runtime** &rarr; **Change runtime type** &rarr; Select **T4 GPU** &rarr; **Save**.

In [ ]:
# 1. Verify GPU Allocation
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Please switch runtime to T4 GPU via Runtime > Change runtime type.")
!nvidia-smi

In [ ]:
# 2. Clone or Update the latest GitHub repository
import os
if not os.path.exists('/content/DIP-project'):
    !git clone https://github.com/Duljit2006/DIP-project.git /content/DIP-project

%cd /content/DIP-project
!git checkout main
!git pull origin main

In [ ]:
# 3. Install PyTorch Geometric and required packages
!pip install -q torch_geometric
!pip install -q scipy opencv-python Pillow matplotlib scikit-image
print("Environment setup complete!")

---
## 1. Quick Test on Single Image
Runs inference through all 4 pipelines on Lena (`img_009_SRF_2.png`) and outputs the 5-panel comparison.

In [ ]:
# 4. Evaluate single image
!python evaluate.py --image datasets/Set14/img_009_SRF_2.png

In [ ]:
# 5. Display Single Image 5-Panel Comparison & 4-Panel Error Heatmap
from IPython.display import Image, display
import os

comp_img = "results/single_img_009_SRF_2_comparison.png"
heat_img = "results/single_img_009_SRF_2_error_heatmap.png"

if os.path.exists(comp_img):
    print("=== 5-Panel Visual Reconstruction (Original vs JPEG vs GNN vs GNN->CNN vs CNN->GNN) ===")
    display(Image(comp_img, width=1200))
if os.path.exists(heat_img):
    print("=== 4-Panel Error Heatmap (Darker = Lowest Error) ===")
    display(Image(heat_img, width=1200))

---
## 2. Full Benchmark Evaluation (Set14 + Set5)
Runs benchmark evaluation across all test images and outputs comparative PSNR/SSIM scores.

In [ ]:
# 6. Run full benchmark evaluation across Set14
!python evaluate.py --dataset Set14

In [ ]:
# 7. Display Summary Evaluation Scores
if os.path.exists('results/final_evaluation_scores.txt'):
    with open('results/final_evaluation_scores.txt') as f:
        print(f.read())

In [ ]:
# 8. Display Zebra (img_013_SRF_2) and PPT3 Text (img_011_SRF_2)
zebra_comp = "results/Set14_img_013_SRF_2_comparison.png"
zebra_heat = "results/Set14_img_013_SRF_2_error_heatmap.png"
if os.path.exists(zebra_comp):
    print("=== Zebra (img_013_SRF_2 - Repeating Textures) ===")
    display(Image(zebra_comp, width=1200))
if os.path.exists(zebra_heat):
    display(Image(zebra_heat, width=1200))

ppt_comp = "results/Set14_img_011_SRF_2_comparison.png"
if os.path.exists(ppt_comp):
    print("=== PPT3 Text (img_011_SRF_2 - Sharp Edges) ===")
    display(Image(ppt_comp, width=1200))

---
## 3. (Optional) Retrain Full Model from Scratch on Colab T4 GPU

In [ ]:
# 9. Retrain Stage 1 (Multi-Block GNN) + Stage 2 (Spatial CNN) on 480,000 blocks
!python train.py --train_folder datasets/BSDS500/train --epochs 25 --cnn_epochs 15 --batch_size 256